In [1]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [41]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        return torch.argmax(logits, dim=-1).item()

    def random_sampling(self, logits: torch.Tensor) -> int:
        probs = F.softmax(logits, dim=-1)
        
        return torch.multinomial(probs, num_samples=1).item()

    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int
    ) -> str:
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt")
        beams = [(0.0, input_ids)]
        
        with torch.no_grad():
            for _ in range(max_length):
                candidates = []
                for score, seq in beams:
                    if seq[0, -1].item() == self.tokenizer.eos_token_id:
                        candidates.append((score, seq))
                        continue
                    
                    outputs = self.model(seq)
                    next_token_probs = F.log_softmax(outputs.logits[0, -1, :], dim=-1)
                    
                    sorted_scores, sorted_indices = torch.sort(next_token_probs, descending=True)
                    
                    best_scores = sorted_scores[:num_beams]
                    best_indices = sorted_indices[:num_beams]
                    
                    for i in range(num_beams):
                        new_score = score + best_scores[i].item()
                        new_seq = torch.cat([seq, torch.tensor([[best_indices[i].item()]])], dim=1)
                        candidates.append((new_score, new_seq))
                
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:num_beams]

        best_seq = beams[0][1]
        
        return self.tokenizer.decode(best_seq[0])

    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        if temperature > 0 and temperature != 1.0:
            logits = logits / temperature
            
        return self.random_sampling(logits)

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
            
        cumulative_prob = 0.0
        cutoff_index = len(logits)

        for i in range(len(sorted_probs)):
            cumulative_prob += sorted_probs[i].item()
            if cumulative_prob > top_p:
                cutoff_index = i + 1 
                break
            
        top_probs = sorted_probs[:cutoff_index]
        top_indices = sorted_indices[:cutoff_index]
        top_probs = top_probs / top_probs.sum()

        sample_idx = torch.multinomial(top_probs, num_samples=1).item()
            
        return top_indices[sample_idx].item()

    def _apply_top_k(self, logits: torch.Tensor, top_k: int = 0) -> torch.Tensor:
        sorted_vals, sorted_inds = torch.sort(logits, descending=True)
        top_vals = sorted_vals[:top_k]
        top_inds = sorted_inds[:top_k]
        probs = F.softmax(top_vals, dim=-1)
        sample_idx = torch.multinomial(probs, num_samples=1).item()
        real_token_id = top_inds[sample_idx].item()
        
        return real_token_id
        
    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        if strategy == "beam":
            return self._beam_search_generate(prompt, max_length, num_beams)

        input_ids = self.tokenizer.encode(prompt, return_tensors="pt")
        ids = input_ids[0].tolist()

        with torch.no_grad():
            for _ in range(max_length):
                outputs = self.model(input_ids)
                logits = outputs.logits[0, -1, :]
                
                next_token_id = None

                if strategy == "greedy":
                    next_token_id = self.greedy_sampling(logits)

                elif strategy == "temperature":
                    next_token_id = self.apply_temperature(logits, temperature)

                elif strategy == "top_k":
                    next_token_id = self._apply_top_k(logits, top_k)

                elif strategy == "top_p":
                    next_token_id = self._apply_top_p(logits, top_p)

                ids.append(next_token_id)
                next_token = torch.tensor([[next_token_id]])
                input_ids = torch.cat([input_ids, next_token], dim=1)

                if next_token_id == self.tokenizer.eos_token_id:
                    break

        return self.tokenizer.decode(ids)

In [45]:
def run_tests():
    model = Model()
    prompt = "Attempt"
    print(f"Prompt: '{prompt}'")

    print("1. Greedy:")
    print(model.generate(prompt, max_length=30, strategy="greedy"))

    print("2. Beam Search:")
    print(model.generate(prompt, max_length=30, strategy="beam", num_beams=3))

    print("3. Temperature (t=1.5):")
    print(model.generate(prompt, max_length=30, strategy="temperature", temperature=1.5))

    print("4. Top-K (k=5):")
    print(model.generate(prompt, max_length=30, strategy="top_k", top_k=5))

    print("5. Top-P (p=0.8):")
    print(model.generate(prompt, max_length=30, strategy="top_p", top_p=0.8))


run_tests()

Prompt: 'Attempt'
1. Greedy:
Attempt to load the file 'C:\Program Files (x86)\Steam\steamapps\workshop\content\211820\7395567'
2. Beam Search:
Attempt.AddEventListener(Unknown Source) at org.luaj.vm2.LuaClosure.invoke(Unknown Source) at org.luaj.vm
3. Temperature (t=1.5):
Attempt 3), noise technology down meets die drop driver Desktop audio replacement 60240 Fortif yields 3 budget sized shareMaybe ain it never hurt movinal electronics younger
4. Top-K (k=5):
Attempt, a group of people who are not members of the group, decided to take over the house and take over the house. It was a very peaceful
5. Top-P (p=0.8):
Attempt, after a number of tragedies in the U.S. – including several since she had taken over as chief medical officer at University of Minnesota – including
